# LlamaIndex: Zero to Advanced
### A comprehensive, hands-on notebook using **Groq** as the LLM backend

This notebook takes you from installing LlamaIndex to building production-grade
Retrieval-Augmented Generation (RAG) systems, agents, and event-driven workflows.

**LLM backend:** [Groq](https://groq.com) (fast open-weight model inference via API)
**Embeddings:** Local HuggingFace embeddings (free, no extra API needed)

> ⚠️ You need a free Groq API key from https://console.groq.com/keys to run the LLM cells.
> Get one, then set it as the `GROQ_API_KEY` environment variable (Section 2 shows how).

## Table of Contents
0. Prerequisites & Installation
1. What is LlamaIndex? Core Concepts
2. Setting Up Groq (LLM) and Embeddings
3. Documents & Nodes
4. Loading Data (Readers)
5. Text Splitting / Node Parsing
6. Your First VectorStoreIndex
7. Querying: The Query Engine
8. Response Modes & Response Synthesis
9. Persisting and Loading Indexes
10. Other Index Types (Summary, Keyword, Tree)
11. Retrievers Deep Dive
12. Node Postprocessors & Reranking
13. Customizing the Query Engine
14. Structured Outputs with Pydantic
15. Metadata: Extraction & Filtering
16. Chat Engines
17. Memory Management
18. Prompt Customization
19. Streaming Responses
20. Tools and Function Calling
21. Agents: FunctionAgent / ReAct
22. Multi-Step & Multi-Document Reasoning
23. Router Query Engine
24. Sub-Question Query Engine
25. Query Transformations (HyDE & friends)
26. Advanced Retrieval: Sentence Window & Auto-Merging
27. External Vector Stores (Chroma)
28. Evaluation of RAG Pipelines
29. Observability, Callbacks & Tracing
30. Workflows: Event-Driven Pipelines
31. Building a Multi-Agent System
32. Putting It Together: An End-to-End RAG App
33. Production Best Practices
34. Next Steps & Resources


---
## 0. Prerequisites & Installation

We'll install:
- `llama-index` — the core framework
- `llama-index-llms-groq` — Groq LLM integration
- `llama-index-embeddings-huggingface` — local embedding models
- `llama-index-vector-stores-chroma` — a persistent vector database
- `llama-index-readers-file` — extra file readers
- `chromadb`, `python-dotenv`

This may take a couple of minutes the first time.

In [ ]:
!pip install -q \
    llama-index \
    llama-index-llms-groq \
    llama-index-embeddings-huggingface \
    llama-index-vector-stores-chroma \
    llama-index-readers-file \
    chromadb \
    python-dotenv \
    groq

print("Installation complete.")

---
## 1. What is LlamaIndex? Core Concepts

**LlamaIndex** is a data framework for connecting large language models (LLMs) to your
own data, so you can build applications like:

- **RAG (Retrieval-Augmented Generation)** chatbots over your documents
- **Agents** that use tools and take actions
- **Structured data extraction** pipelines
- **Multi-agent workflows**

### The core pipeline

```
Raw Data --> Documents --> Nodes --> Index --> Retriever --> Query Engine --> Response
   (files)    (loaded)    (chunks)  (vector DB)  (search)     (LLM + context)
```

| Concept | What it is |
|---|---|
| **Document** | A container for a piece of raw data (a PDF, webpage, etc.) plus metadata |
| **Node** | A chunk of a Document — the atomic unit that gets embedded and retrieved |
| **Index** | A data structure (usually a vector store) that organizes Nodes for fast retrieval |
| **Retriever** | Fetches the most relevant Nodes for a query |
| **Query Engine** | Retriever + LLM: retrieves context, then asks the LLM to answer using it |
| **Chat Engine** | Like a query engine, but keeps conversational memory |
| **Agent** | An LLM that can decide to call tools/functions to accomplish a task |

We'll build up through every layer of this stack.

---
## 2. Setting Up Groq (LLM) and Embeddings

### 2.1 Get a Groq API key
1. Go to https://console.groq.com/keys
2. Create a free API key
3. Set it as an environment variable (recommended: use a `.env` file)

### 2.2 Choosing a Groq model
Groq hosts several open-weight models with extremely fast inference. Common choices (check
https://console.groq.com/docs/models for the current list, model names change over time):
- `llama-3.3-70b-versatile` — strong general-purpose model
- `llama-3.1-8b-instant` — fast & cheap for simple tasks
- `gemma2-9b-it` — Google's Gemma model

### 2.3 Embeddings
Groq does **not** serve embedding models, so we use a free local embedding model from
HuggingFace (`BAAI/bge-small-en-v1.5`) that runs on CPU. This never leaves your machine
and has no rate limits.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads GROQ_API_KEY from a .env file if present

# If you don't use a .env file, set it directly (not recommended to hardcode in shared notebooks):
# os.environ["GROQ_API_KEY"] = "gsk_..."

assert os.environ.get("GROQ_API_KEY"), "Please set the GROQ_API_KEY environment variable."
print("Groq API key found.")

In [ ]:
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# 1. Configure the LLM
llm = Groq(model="llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])

# 2. Configure local embeddings (downloads the model once, then caches it)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 3. Register both as GLOBAL defaults so every index/query engine uses them automatically
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 50

# Quick sanity check
response = llm.complete("In one sentence, what is Retrieval-Augmented Generation?")
print(response)

---
## 3. Documents & Nodes

A **Document** wraps raw text + metadata. A **Node** is a chunk of a Document — the unit
that actually gets embedded and stored. You can create these manually or let LlamaIndex
generate them for you from files (next section).

In [ ]:
from llama_index.core import Document
from llama_index.core.schema import TextNode

# Creating a Document manually
doc = Document(
    text="LlamaIndex is a data framework for LLM applications. "
         "It helps you ingest, structure, and access private or domain-specific data.",
    metadata={"source": "manual_example", "topic": "llamaindex_intro"},
)
print("Document ID:", doc.doc_id)
print("Metadata:", doc.metadata)

# Creating Nodes manually (usually done automatically by a NodeParser, see Section 5)
node1 = TextNode(text="LlamaIndex helps connect LLMs to your data.", id_="node-1")
node2 = TextNode(text="It supports RAG, agents, and structured extraction.", id_="node-2")
print(node1)

---
## 4. Loading Data (Readers)

LlamaIndex ships **Readers** (aka "loaders") for dozens of data sources: local files,
PDFs, Notion, Slack, databases, websites, and more (see [LlamaHub](https://llamahub.ai)).

The simplest one is `SimpleDirectoryReader`, which loads every supported file in a folder.
Let's create a tiny sample dataset to use throughout the notebook.

In [ ]:
import os

os.makedirs("data", exist_ok=True)

with open("data/llamaindex_overview.txt", "w") as f:
    f.write('''LlamaIndex Overview

LlamaIndex is an open-source data framework designed to connect large language models (LLMs)
to external data sources. It provides tools for data ingestion, indexing, and querying,
enabling developers to build powerful applications such as chatbots, question-answering
systems, and autonomous agents that can reason over private data.

Core components include Documents, Nodes, Indexes, Retrievers, and Query Engines. LlamaIndex
integrates with many vector databases, LLM providers (including Groq), and embedding models.
''')

with open("data/rag_concepts.txt", "w") as f:
    f.write('''Retrieval-Augmented Generation (RAG)

RAG is a technique that combines information retrieval with text generation. Instead of
relying solely on an LLM's training data, RAG systems first retrieve relevant chunks of
text from an external knowledge base, then pass that context to the LLM to generate a
grounded, up-to-date answer. This reduces hallucination and allows LLMs to answer questions
about private or recent data they were never trained on.

Key steps in a RAG pipeline: load data, split into chunks (nodes), embed the chunks,
store them in a vector index, retrieve relevant chunks for a query, and synthesize a
final answer using an LLM.
''')

print("Sample data files created in ./data")

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("data").load_data()
print(f"Loaded {len(documents)} document(s)")
print(documents[0].text[:200], "...")
print(documents[0].metadata)

---
## 5. Text Splitting / Node Parsing

Documents are usually too big to embed as a single chunk. A **NodeParser** (aka text
splitter) breaks them into smaller **Nodes**. LlamaIndex offers several strategies:

- `SentenceSplitter` — splits on sentence boundaries, respecting a target chunk size (default, good general choice)
- `TokenTextSplitter` — splits by raw token count
- `SemanticSplitterNodeParser` — splits based on semantic similarity between sentences (uses embeddings)
- `HierarchicalNodeParser` — creates parent/child chunks at multiple granularities (used for auto-merging retrieval, Section 26)

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=200, chunk_overlap=20)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} nodes")
for n in nodes[:3]:
    print("---")
    print(n.text[:150])

---
## 6. Your First VectorStoreIndex

A `VectorStoreIndex` embeds every Node and stores the vectors for fast similarity search.
By default it uses an in-memory vector store — perfect for learning and prototyping.

In [ ]:
from llama_index.core import VectorStoreIndex

# Option A: build directly from Documents (LlamaIndex parses+embeds internally)
index = VectorStoreIndex.from_documents(documents)

# Option B: build from Nodes you already created (more control)
# index = VectorStoreIndex(nodes)

print("Index built with", len(index.docstore.docs), "nodes")

---
## 7. Querying: The Query Engine

A **Query Engine** wraps a Retriever + LLM. `.as_query_engine()` gives you a one-liner
RAG pipeline: embed the query → retrieve top-k similar nodes → stuff them into a prompt →
ask the LLM.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("What is RAG and why is it useful?")
print(response)

In [ ]:
# Inspect exactly which chunks were used to ground the answer
for i, node_with_score in enumerate(response.source_nodes):
    print(f"--- Source {i+1} (score={node_with_score.score:.3f}) ---")
    print(node_with_score.node.text[:200])
    print()

---
## 8. Response Modes & Response Synthesis

Once nodes are retrieved, LlamaIndex needs to turn them into a final answer. This step is
the **response synthesizer**, controlled by `response_mode`:

| Mode | Behavior |
|---|---|
| `compact` (default) | Stuffs as many chunks as fit into one prompt, minimizing LLM calls |
| `refine` | Iteratively refines the answer, one chunk at a time (higher quality, slower, more calls) |
| `tree_summarize` | Recursively summarizes chunks in a tree, good for big context / summarization |
| `simple_summarize` | Truncates and does a single summarization call |
| `no_text` | Returns only retrieved nodes, no LLM call (useful for debugging retrieval) |
| `accumulate` | Runs the query against each chunk independently and concatenates answers |

In [ ]:
for mode in ["compact", "refine", "tree_summarize"]:
    qe = index.as_query_engine(response_mode=mode, similarity_top_k=3)
    resp = qe.query("Summarize what LlamaIndex is used for.")
    print(f"=== mode: {mode} ===")
    print(resp)
    print()

---
## 9. Persisting and Loading Indexes

Rebuilding (and re-embedding) an index every time is wasteful. Persist it to disk once,
then load it instantly later.

In [ ]:
PERSIST_DIR = "./storage"

# Save
index.storage_context.persist(persist_dir=PERSIST_DIR)
print("Index persisted to", PERSIST_DIR)

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
loaded_index = load_index_from_storage(storage_context)

qe = loaded_index.as_query_engine()
print(qe.query("What is LlamaIndex?"))

---
## 10. Other Index Types

`VectorStoreIndex` is the workhorse, but LlamaIndex offers other index structures for
different use cases:

- **`SummaryIndex`** (formerly `ListIndex`) — stores nodes as a simple sequential list.
  No embeddings needed; queries can iterate over *all* nodes (great for pure summarization
  of small corpora, or when you want guaranteed full coverage).
- **`TreeIndex`** — builds a hierarchical tree of summaries, useful for summarizing very
  large documents efficiently.
- **`KeywordTableIndex`** — extracts keywords per node and retrieves via keyword match
  instead of embeddings (fast, no embedding cost, weaker semantic matching).

Which one to use depends on your data size and query patterns — vector search dominates
in practice, but these are useful specialized tools.

In [ ]:
from llama_index.core import SummaryIndex, KeywordTableIndex

summary_index = SummaryIndex.from_documents(documents)
summary_qe = summary_index.as_query_engine(response_mode="tree_summarize")
print("SummaryIndex answer:")
print(summary_qe.query("Give a one-paragraph summary of all the documents."))
print()

keyword_index = KeywordTableIndex.from_documents(documents)
keyword_qe = keyword_index.as_query_engine()
print("KeywordTableIndex answer:")
print(keyword_qe.query("What does RAG stand for?"))

---
## 11. Retrievers Deep Dive

The **Retriever** is the component responsible for fetching relevant Nodes. You can use a
retriever standalone (without an LLM call) to inspect exactly what would be retrieved, or
combine retrievers, or write a custom one.

In [ ]:
retriever = index.as_retriever(similarity_top_k=2)
retrieved_nodes = retriever.retrieve("What is a Node in LlamaIndex?")

for n in retrieved_nodes:
    print(f"score={n.score:.3f}")
    print(n.node.text[:150])
    print("---")

In [ ]:
from llama_index.core.retrievers import QueryFusionRetriever

# Example: combine a vector retriever with a keyword retriever, fusing results
vector_retriever = index.as_retriever(similarity_top_k=3)
keyword_retriever = keyword_index.as_retriever()

fusion_retriever = QueryFusionRetriever(
    [vector_retriever, keyword_retriever],
    similarity_top_k=3,
    num_queries=1,  # set >1 to auto-generate query variations with the LLM
    mode="reciprocal_rerank",
)

results = fusion_retriever.retrieve("What are the core components of LlamaIndex?")
for r in results:
    print(round(r.score, 3), "-", r.node.text[:100])

---
## 12. Node Postprocessors & Reranking

**Postprocessors** run *after* retrieval and *before* synthesis — filtering, reordering,
or re-scoring nodes. Common ones:

- `SimilarityPostprocessor` — drops nodes below a similarity cutoff
- `KeywordNodePostprocessor` — requires/excludes certain keywords
- `SentenceEmbeddingOptimizer` — trims each node down to only the most relevant sentences
- Rerankers (e.g. cross-encoder models, Cohere Rerank) — re-score retrieved nodes with a
  more accurate (but slower) model for higher precision

In [ ]:
from llama_index.core.postprocessor import SimilarityPostprocessor

postprocessor = SimilarityPostprocessor(similarity_cutoff=0.5)

query_engine = index.as_query_engine(
    similarity_top_k=5,
    node_postprocessors=[postprocessor],
)
response = query_engine.query("What is RAG?")
print(response)
print(f"\n{len(response.source_nodes)} node(s) survived the similarity cutoff")

---
## 13. Customizing the Query Engine

Under the hood, `.as_query_engine()` assembles a `RetrieverQueryEngine` from a retriever
and a response synthesizer. You can build this manually for full control.

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

retriever = index.as_retriever(similarity_top_k=3)
synthesizer = get_response_synthesizer(response_mode="compact")

custom_query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.4)],
)

print(custom_query_engine.query("Explain how a query engine works."))

---
## 14. Structured Outputs with Pydantic

LLMs are great at free text, but applications often need structured data. LlamaIndex can
force the LLM to return a validated Pydantic object.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class KeyConcept(BaseModel):
    """A key concept extracted from text."""
    name: str = Field(description="Name of the concept")
    definition: str = Field(description="One-sentence definition")

class ConceptList(BaseModel):
    concepts: List[KeyConcept]

sllm = llm.as_structured_llm(ConceptList)
resp = sllm.complete(
    "Extract the key concepts from this text: "
    + documents[0].text
)
result: ConceptList = resp.raw
for c in result.concepts:
    print(f"- {c.name}: {c.definition}")

---
## 15. Metadata: Extraction & Filtering

Metadata (source, date, author, section, page number, custom tags...) can be attached to
nodes and used to **filter** retrieval — e.g., "only search documents tagged `topic=rag`".
LlamaIndex can also auto-generate metadata (titles, summaries, questions-this-can-answer)
using the LLM via `Extractor` modules.

In [ ]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

filters = MetadataFilters(filters=[ExactMatchFilter(key="file_name", value="rag_concepts.txt")])

filtered_qe = index.as_query_engine(filters=filters, similarity_top_k=3)
print(filtered_qe.query("What does this document say?"))

In [ ]:
from llama_index.core.extractors import TitleExtractor, QuestionsAnsweredExtractor
from llama_index.core.ingestion import IngestionPipeline

# Automatically enrich nodes with an LLM-generated title and Q&A pairs they can answer
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=20),
        TitleExtractor(nodes=3, llm=llm),
        QuestionsAnsweredExtractor(questions=2, llm=llm),
    ]
)

enriched_nodes = pipeline.run(documents=documents)
print(enriched_nodes[0].metadata)

---
## 16. Chat Engines

A **Query Engine** is stateless — every call is independent. A **Chat Engine** keeps
conversation history so follow-up questions ("what about *that*?") work correctly.
Common chat modes:

- `condense_question` — rewrites the follow-up into a standalone question, then does RAG
- `context` — always retrieves fresh context and injects it alongside chat history
- `condense_plus_context` — combination of both (usually the best default)
- `react` — a ReAct agent that can also use tools while chatting (see Section 21)

In [ ]:
chat_engine = index.as_chat_engine(chat_mode="condense_plus_context", llm=llm)

print(chat_engine.chat("What is LlamaIndex?"))
print()
print(chat_engine.chat("What did I just ask you about?"))  # tests memory

In [ ]:
chat_engine.reset()  # clear conversation history when starting a new session

---
## 17. Memory Management

By default, chat engines use a `ChatMemoryBuffer` that keeps the last N tokens of history.
You can configure it explicitly, or plug in other memory types for longer conversations
(e.g., summarizing old turns instead of dropping them).

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(token_limit=3000)

chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=memory,
    llm=llm,
)
print(chat_engine.chat("Hi, I'm exploring LlamaIndex for a university project."))
print(chat_engine.chat("What should I focus on first?"))

---
## 18. Prompt Customization

Every LlamaIndex module (query engines, chat engines, extractors...) uses internal prompt
templates you can inspect and override — useful for tone, language, or format control.

In [ ]:
prompts_dict = query_engine.get_prompts()
for name, prompt in prompts_dict.items():
    print(f"### {name}")
    print(prompt.get_template()[:300])
    print()

In [ ]:
from llama_index.core import PromptTemplate

custom_qa_prompt = PromptTemplate(
    "You are a concise technical assistant. Using ONLY the context below, "
    "answer the question in at most 2 sentences.\n"
    "Context:\n{context_str}\n"
    "Question: {query_str}\n"
    "Answer:"
)

query_engine.update_prompts({"response_synthesizer:text_qa_template": custom_qa_prompt})
print(query_engine.query("What is a Node?"))

---
## 19. Streaming Responses

For chat UIs, you usually want tokens to appear as they're generated rather than waiting
for the full response.

In [ ]:
streaming_qe = index.as_query_engine(streaming=True, similarity_top_k=3)
streaming_response = streaming_qe.query("Explain RAG in simple terms.")

for token in streaming_response.response_gen:
    print(token, end="", flush=True)

---
## 20. Tools and Function Calling

**Tools** let an LLM call out to arbitrary Python functions — the foundation of agents.
LlamaIndex wraps any function as a `FunctionTool`, auto-generating its schema from the
type hints and docstring.

> Note: tool/function calling requires a Groq model that supports it (e.g.
> `llama-3.3-70b-versatile`, `llama-3.1-8b-instant`).

In [ ]:
from llama_index.core.tools import FunctionTool

def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b

def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

multiply_tool = FunctionTool.from_defaults(fn=multiply)
add_tool = FunctionTool.from_defaults(fn=add)

response = llm.predict_and_call([multiply_tool, add_tool], "What is (3 + 5) multiplied by 7?")
print(response)

---
## 21. Agents: FunctionAgent / ReAct

An **Agent** wraps an LLM in a loop: think → decide whether to call a tool → observe the
result → repeat → final answer. LlamaIndex provides:

- `FunctionAgent` — uses native function/tool calling (best when the model supports it, like Groq's Llama 3.x models)
- `ReActAgent` — uses the ReAct prompting pattern (Reason + Act), works with any chat model, even without native tool-calling support

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
import asyncio

agent = FunctionAgent(
    tools=[multiply_tool, add_tool],
    llm=llm,
    system_prompt="You are a helpful math assistant. Use tools for any arithmetic.",
)

async def run_agent():
    response = await agent.run("What is 12 multiplied by 4, then plus 100?")
    return response

result = asyncio.run(run_agent())
print(result)

In [ ]:
# You can also give an agent a RAG query engine AS a tool, so it can decide when to
# search your documents vs. answer directly.
from llama_index.core.tools import QueryEngineTool

rag_tool = QueryEngineTool.from_defaults(
    query_engine=index.as_query_engine(),
    name="llamaindex_knowledge_base",
    description="Answers questions about LlamaIndex concepts using the loaded documents.",
)

rag_agent = FunctionAgent(
    tools=[rag_tool, multiply_tool],
    llm=llm,
    system_prompt="Answer using the knowledge base tool when the question is about LlamaIndex.",
)

async def run_rag_agent():
    return await rag_agent.run("According to the knowledge base, what is a Node?")

print(asyncio.run(run_rag_agent()))

---
## 22. Multi-Step & Multi-Document Reasoning

Real questions often need *multiple* retrieval/reasoning steps ("compare X and Y", "first
find A, then use A to look up B"). Agents naturally handle this because they loop until
done. You can also explicitly decompose a complex query into sub-questions — covered next
with the Sub-Question Query Engine (Section 24).

In [ ]:
async def run_multistep():
    return await rag_agent.run(
        "First look up what a Node is in the knowledge base, then explain how that "
        "differs from a Document, using your own words."
    )

print(asyncio.run(run_multistep()))

---
## 23. Router Query Engine

When you have several specialized query engines (e.g., one per index type or per data
source), a **RouterQueryEngine** uses the LLM to pick the best one for each incoming
query, based on natural-language descriptions you provide.

In [ ]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

vector_tool = QueryEngineTool.from_defaults(
    query_engine=index.as_query_engine(),
    description="Good for specific factual questions about LlamaIndex concepts.",
)
summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_index.as_query_engine(response_mode="tree_summarize"),
    description="Good for broad summarization requests across all documents.",
)

router_qe = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(llm=llm),
    query_engine_tools=[vector_tool, summary_tool],
)

print(router_qe.query("Summarize everything in the knowledge base."))
print()
print(router_qe.query("What specifically is a Node?"))

---
## 24. Sub-Question Query Engine

For compound questions, the **Sub-Question Query Engine** first asks the LLM to break the
question into simpler sub-questions, answers each one against the right tool, then
synthesizes a final combined answer. Great for comparisons or multi-part questions.

In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine

sub_question_qe = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[vector_tool, summary_tool],
    llm=llm,
)

response = sub_question_qe.query(
    "What is a Node, and separately, what is RAG used for?"
)
print(response)

---
## 25. Query Transformations (HyDE & friends)

Sometimes the raw user query is a poor match for embedding-based search (too short, oddly
phrased). **Query transformations** rewrite the query before retrieval:

- **HyDE** (Hypothetical Document Embeddings) — asks the LLM to write a *hypothetical
  answer* first, then embeds *that* (hypothetical answers often resemble real chunks more
  closely than short questions do)
- Multi-step transforms — decompose a complex query into a sequence of simpler ones

In [ ]:
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine

hyde = HyDEQueryTransform(llm=llm, include_original=True)
hyde_query_engine = TransformQueryEngine(index.as_query_engine(), query_transform=hyde)

print(hyde_query_engine.query("core components"))

---
## 26. Advanced Retrieval: Sentence Window & Auto-Merging

Two techniques that improve RAG quality beyond naive fixed-size chunking:

- **Sentence Window Retrieval**: index individual *sentences*, but when one is retrieved,
  expand it to include a window of surrounding sentences before sending to the LLM —
  precise matching, rich context.
- **Auto-Merging Retrieval**: build a hierarchy of chunks (small → medium → large). If
  enough small child chunks under the same parent are retrieved, automatically "merge" them
  and return the larger parent chunk instead — avoids fragmented context.

In [ ]:
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

window_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)
window_nodes = window_parser.get_nodes_from_documents(documents)
window_index = VectorStoreIndex(window_nodes)

window_qe = window_index.as_query_engine(
    similarity_top_k=2,
    node_postprocessors=[MetadataReplacementPostProcessor(target_metadata_key="window")],
)
print(window_qe.query("What does LlamaIndex integrate with?"))

In [ ]:
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core import StorageContext
from llama_index.core.query_engine import RetrieverQueryEngine

hierarchical_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[512, 256, 128])
hierarchical_nodes = hierarchical_parser.get_nodes_from_documents(documents)
leaf_nodes = get_leaf_nodes(hierarchical_nodes)

storage_context = StorageContext.from_defaults()
storage_context.docstore.add_documents(hierarchical_nodes)  # store ALL levels (needed for merging)

automerge_index = VectorStoreIndex(leaf_nodes, storage_context=storage_context)
base_retriever = automerge_index.as_retriever(similarity_top_k=6)
automerge_retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True)

automerge_qe = RetrieverQueryEngine.from_args(automerge_retriever)
print(automerge_qe.query("What is LlamaIndex used for?"))

---
## 27. External Vector Stores (Chroma)

The default in-memory vector store is fine for learning, but production apps need a
persistent, scalable vector database. LlamaIndex integrates with 20+ vector stores
(Chroma, Pinecone, Qdrant, Weaviate, pgvector, Milvus...). Here's Chroma, which runs
locally with zero setup.

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_or_create_collection("llamaindex_demo")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

chroma_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(chroma_index.as_query_engine().query("What is LlamaIndex?"))

In [ ]:
# Later (even in a new session), reconnect without re-embedding:
chroma_client2 = chromadb.PersistentClient(path="./chroma_db")
collection2 = chroma_client2.get_or_create_collection("llamaindex_demo")
vector_store2 = ChromaVectorStore(chroma_collection=collection2)

reloaded_chroma_index = VectorStoreIndex.from_vector_store(vector_store2)
print(reloaded_chroma_index.as_query_engine().query("What does RAG stand for?"))

---
## 28. Evaluation of RAG Pipelines

You can't improve what you don't measure. LlamaIndex has built-in LLM-as-judge evaluators:

- **FaithfulnessEvaluator** — does the answer stay grounded in the retrieved context (no hallucination)?
- **RelevancyEvaluator** — is the answer actually relevant to the query?
- **CorrectnessEvaluator** — how correct is the answer vs. a reference answer?
- **Retrieval evaluators** (`hit_rate`, `mrr`) — measure retriever quality directly against a labeled dataset

In [ ]:
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

faithfulness_evaluator = FaithfulnessEvaluator(llm=llm)
relevancy_evaluator = RelevancyEvaluator(llm=llm)

query = "What is a Node in LlamaIndex?"
response = query_engine.query(query)

faith_result = faithfulness_evaluator.evaluate_response(response=response)
rel_result = relevancy_evaluator.evaluate_response(query=query, response=response)

print("Faithful (grounded in context)?", faith_result.passing)
print("Relevant to the query?", rel_result.passing)

In [ ]:
from llama_index.core.evaluation import generate_question_context_pairs, RetrieverEvaluator
import asyncio

# Auto-generate an evaluation dataset of (question -> expected node) pairs from your own nodes
qa_dataset = generate_question_context_pairs(nodes, llm=llm, num_questions_per_chunk=1)

retriever_evaluator = RetrieverEvaluator.from_metric_names(
    ["hit_rate", "mrr"], retriever=index.as_retriever(similarity_top_k=3)
)

eval_results = asyncio.run(retriever_evaluator.aevaluate_dataset(qa_dataset))
hit_rates = [r.metric_vals_dict["hit_rate"] for r in eval_results]
mrrs = [r.metric_vals_dict["mrr"] for r in eval_results]
print(f"Average hit rate: {sum(hit_rates)/len(hit_rates):.2f}")
print(f"Average MRR: {sum(mrrs)/len(mrrs):.2f}")

---
## 29. Observability, Callbacks & Tracing

For debugging and production monitoring, LlamaIndex supports a `CallbackManager` that can
log every LLM call, embedding call, and retrieval event — and integrates with tracing
tools (Arize Phoenix, LangSmith-style tracing, W&B, etc.).

In [ ]:
from llama_index.core.callbacks import CallbackManager, LlamaDebugHandler

debug_handler = LlamaDebugHandler(print_trace_on_end=True)
callback_manager = CallbackManager([debug_handler])

Settings.callback_manager = callback_manager

debug_qe = index.as_query_engine()
debug_qe.query("What is LlamaIndex?")

# Inspect captured events
print(debug_handler.get_event_pairs())[:1]

# Reset for the rest of the notebook
Settings.callback_manager = CallbackManager([])

---
## 30. Workflows: Event-Driven Pipelines

`llama_index.core.workflow` is the modern, low-level way to build custom multi-step LLM
applications (agents, pipelines, RAG with branching logic) as an explicit **graph of
steps connected by typed events** — more flexible and debuggable than a fixed chain.

Core building blocks:
- **`Workflow`** — the overall pipeline
- **`@step`** — decorator marking a method as one node in the graph
- **`Event`** subclasses — typed messages passed between steps
- **`StartEvent` / `StopEvent`** — built-in entry/exit events

In [ ]:
from llama_index.core.workflow import (
    Workflow, step, Event, StartEvent, StopEvent, Context
)

class RetrieveEvent(Event):
    query: str

class SynthesizeEvent(Event):
    query: str
    context_str: str

class SimpleRAGWorkflow(Workflow):

    @step
    async def route(self, ev: StartEvent) -> RetrieveEvent:
        return RetrieveEvent(query=ev.query)

    @step
    async def retrieve(self, ev: RetrieveEvent) -> SynthesizeEvent:
        nodes = index.as_retriever(similarity_top_k=3).retrieve(ev.query)
        context_str = "\n\n".join(n.node.text for n in nodes)
        return SynthesizeEvent(query=ev.query, context_str=context_str)

    @step
    async def synthesize(self, ev: SynthesizeEvent) -> StopEvent:
        prompt = (
            f"Context:\n{ev.context_str}\n\n"
            f"Question: {ev.query}\nAnswer concisely using only the context."
        )
        response = await llm.acomplete(prompt)
        return StopEvent(result=str(response))

workflow = SimpleRAGWorkflow(timeout=60, verbose=False)

async def run_workflow():
    return await workflow.run(query="What is a Document in LlamaIndex?")

print(asyncio.run(run_workflow()))

---
## 31. Building a Multi-Agent System

Complex tasks often benefit from *specialized* agents collaborating — e.g., a "Researcher"
agent that searches the knowledge base, and a "Writer" agent that drafts the final answer.
`AgentWorkflow` orchestrates handoffs between multiple `FunctionAgent`s automatically.

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

researcher = FunctionAgent(
    name="Researcher",
    description="Searches the knowledge base for relevant facts.",
    tools=[rag_tool],
    llm=llm,
    system_prompt="Find relevant facts from the knowledge base and hand off to the Writer.",
    can_handoff_to=["Writer"],
)

writer = FunctionAgent(
    name="Writer",
    description="Writes a clear final answer from researched facts.",
    tools=[],
    llm=llm,
    system_prompt="Write a concise, well-structured final answer using the facts you receive.",
)

multi_agent_workflow = AgentWorkflow(
    agents=[researcher, writer],
    root_agent="Researcher",
)

async def run_multi_agent():
    return await multi_agent_workflow.run(
        user_msg="Explain, in a short paragraph, what a Node and a Document are and how they relate."
    )

print(asyncio.run(run_multi_agent()))

---
## 32. Putting It Together: An End-to-End RAG App

Let's assemble everything into one reusable pipeline: load → parse → embed → persist to
Chroma → retrieve with postprocessing → chat with memory → evaluate.

In [ ]:
class RAGApp:
    def __init__(self, data_dir: str, persist_path: str = "./chroma_db_app", collection: str = "app_docs"):
        self.llm = Groq(model="llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
        self.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
        Settings.llm = self.llm
        Settings.embed_model = self.embed_model

        client = chromadb.PersistentClient(path=persist_path)
        chroma_collection = client.get_or_create_collection(collection)
        vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        docs = SimpleDirectoryReader(data_dir).load_data()
        self.index = VectorStoreIndex.from_documents(docs, storage_context=storage_context)

        self.chat_engine = self.index.as_chat_engine(
            chat_mode="condense_plus_context",
            llm=self.llm,
            memory=ChatMemoryBuffer.from_defaults(token_limit=3000),
        )
        self.evaluator = FaithfulnessEvaluator(llm=self.llm)

    def ask(self, question: str, evaluate: bool = True):
        response = self.chat_engine.chat(question)
        if evaluate:
            result = self.evaluator.evaluate_response(response=response)
            print(f"[faithful: {result.passing}]")
        return response


app = RAGApp(data_dir="data")
print(app.ask("What is LlamaIndex and what is it used for?"))
print(app.ask("How does that relate to RAG?"))

---
## 33. Production Best Practices

- **Chunking**: tune `chunk_size`/`chunk_overlap` per data type; test with real queries, don't guess.
- **Hybrid retrieval**: combine vector + keyword/BM25 search (via `QueryFusionRetriever`) for robustness.
- **Rerank**: add a cross-encoder or hosted reranker after retrieval for higher precision at low extra cost.
- **Cache embeddings**: never re-embed unchanged documents — use `IngestionPipeline` with a persistent cache/docstore.
- **Guard against prompt injection**: treat retrieved content as untrusted; keep tool/agent permissions minimal.
- **Rate limits & retries**: Groq (like all hosted LLM APIs) has rate limits — wrap calls with retry/backoff for production traffic.
- **Evaluate continuously**: keep a small labeled eval set; regress-test retrieval and generation whenever you change chunking, prompts, or models.
- **Observability**: log queries, retrieved sources, and responses; add tracing (Section 29) before you have production incidents, not after.
- **Cost & latency**: Groq's speed is a big advantage here — but still monitor token usage, especially with `refine` mode or big `similarity_top_k`.
- **Security**: never log raw API keys; scope index access to what a given user is allowed to see (metadata filters help here).

---
## 34. Next Steps & Resources

You've now covered the full LlamaIndex stack: data loading, indexing, retrieval,
postprocessing, chat, agents, workflows, evaluation, and production concerns.

**Where to go from here:**
- Official docs: https://docs.llamaindex.ai
- LlamaHub (readers, tools, packs): https://llamahub.ai
- Groq model docs: https://console.groq.com/docs/models
- Try swapping in a real vector store you'd use in production (Qdrant/Pinecone/pgvector)
- Try `llama-index-llms-groq` with a smaller/faster model for latency-sensitive agents, and
  the 70B model for complex reasoning steps
- Build multi-modal RAG (images + text) with `llama-index-multi-modal-llms-*` packages
- Explore `llama-index-packs` for pre-built pipelines (e.g., `ResumeScreenerPack`, `SelfDiscoverPack`)

Happy building! 🦙